# 🚗 Used Car Price Prediction Using XGBoost with Feature Engineering and Gradio

**B.Tech AI & Data Science — Machine Learning Capstone Project**

### Project Overview
Pricing a used car is often based on guesswork and negotiation, which can lead buyers to overpay
and sellers to under- or over-price their vehicle. This project builds a **regression model**
using **XGBoost** that predicts a fair estimated price for a used car based on its brand, **exact
model**, model year, mileage, engine specs, accident history, and other attributes.

The trained model is deployed through a **Gradio** web interface with a **Brand → Model cascading
dropdown** and a one-click **USD → INR** price conversion.

**Type of ML problem:** Supervised Learning → Regression (target variable `price` is continuous).

---
### 🔄 v2 Update Summary
This version updates the original notebook with two new capabilities:
1. **`model` is now a real input feature to XGBoost** (previously excluded due to high cardinality —
   now included using `OneHotEncoder(min_frequency=5)` to control dimensionality).
2. **Gradio now has a cascading Brand → Model dropdown** and a **₹ Convert to INR** button that
   converts the already-predicted USD price without re-running XGBoost.

Everything else (cleaning, other engineered features, XGBoost config, evaluation, graphs) is
preserved from the original notebook.


## Cell 1 — Install Libraries
Run this once per Colab session. *(unchanged)*

In [ ]:
!pip install -q xgboost gradio


## Cell 2 — Import Libraries *(unchanged)*

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

import gradio as gr

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")


## Cell 3 — Upload and Load Dataset *(unchanged)*

Run this cell in Google Colab. It will open a file picker so you can upload the used-cars CSV.
If the file is already sitting in the Colab environment, it will be picked up automatically.

In [ ]:
import os

CSV_PATH = None

possible_local_paths = ["used_cars.csv", "/content/used_cars.csv"]
for p in possible_local_paths:
    if os.path.exists(p):
        CSV_PATH = p
        break

if CSV_PATH is None:
    try:
        from google.colab import files
        print("No local CSV found — please upload the used cars CSV file.")
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            "No CSV found and this is not running inside Google Colab. "
            "Please place 'used_cars.csv' in the working directory."
        )

print("Using dataset file:", CSV_PATH)
df_raw = pd.read_csv(CSV_PATH)
print("Dataset loaded. Shape:", df_raw.shape)


No local CSV found — please upload the used cars CSV file.


## Cell 4 — Inspect Dataset *(unchanged, plus a Brand→Model cardinality check)*

We re-confirm the dataset shape/columns, and specifically look at how many models exist per
brand — this justifies the `min_frequency` choice used later when encoding `model`.

In [ ]:
print("Shape:", df_raw.shape)
print()
print("Columns:", list(df_raw.columns))
print()
print("Data types:")
print(df_raw.dtypes)
print()
print("Missing values per column:")
print(df_raw.isnull().sum())
print()
print("Duplicate rows:", df_raw.duplicated().sum())
print()
print("First 5 rows:")
display(df_raw.head())
print()
print("Statistical summary (numeric columns):")
display(df_raw.describe(include='number'))


In [ ]:
categorical_cols_check = ['brand', 'model', 'fuel_type', 'transmission',
                           'ext_col', 'int_col', 'accident', 'clean_title']
for col in categorical_cols_check:
    print(f"{col:15s} -> {df_raw[col].nunique()} unique values")

print()
# NEW: check how many distinct models exist per brand, and how rare individual models are.
# This tells us that a large number of specific models appear only once or twice in the data,
# which is why we need to group very rare models together during encoding (min_frequency).
models_per_brand = df_raw.groupby('brand')['model'].nunique().sort_values(ascending=False)
print("Models per brand (top 10):")
print(models_per_brand.head(10))

model_counts = df_raw['model'].value_counts()
print()
print(f"Total distinct models: {len(model_counts)}")
print(f"Models appearing only once in the dataset: {(model_counts == 1).sum()}")
print(f"Models appearing 5 times or fewer: {(model_counts <= 5).sum()}")


## Cell 5 — Data Cleaning *(unchanged)*

Same as before: `price` and `milage` are converted from text to numeric, key rows with unparseable
numeric fields are dropped, extreme price outliers (top 1%) are removed, and missing categorical
values are filled with sensible defaults.

In [ ]:
df = df_raw.copy()

# --- Clean price: remove $ and , then convert to float ---
df['price'] = df['price'].astype(str).str.replace(r'[\$,]', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# --- Clean milage: remove everything except digits and decimal point ---
df['milage'] = df['milage'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['milage'] = pd.to_numeric(df['milage'], errors='coerce')

# --- Ensure model_year is numeric ---
df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce')

# --- Drop rows where key numeric fields could not be parsed at all ---
before_rows = df.shape[0]
df = df.dropna(subset=['price', 'milage', 'model_year']).reset_index(drop=True)
print(f"Dropped {before_rows - df.shape[0]} rows with unparseable price/mileage/year.")

# --- Remove extreme price outliers (top 1%) so a few exotic-car listings ---
# --- don't distort the model or the error metrics ------------------------
price_cap = df['price'].quantile(0.99)
before_rows = df.shape[0]
df = df[df['price'] <= price_cap].reset_index(drop=True)
print(f"Removed {before_rows - df.shape[0]} extreme price outliers above ${price_cap:,.0f}.")

# --- Handle missing categorical values ---
df['fuel_type'] = df['fuel_type'].replace('–', np.nan)
df['fuel_type'] = df['fuel_type'].fillna('Unknown')
df['accident'] = df['accident'].fillna('None reported')
df['clean_title'] = df['clean_title'].fillna('No')

# --- 'model' has no missing values in this dataset, but guard against it anyway ---
df['model'] = df['model'].fillna('Unknown')

print("Missing values remaining:")
print(df[['price', 'milage', 'model_year', 'fuel_type', 'accident', 'clean_title', 'model']].isnull().sum())
print()
print("Cleaned shape:", df.shape)
df[['price', 'milage', 'model_year']].head()


## Cell 6 — Feature Engineering *(unchanged)*

Same engineered features as before: `car_age`, `mileage_per_year`, `engine_horsepower`,
`engine_liters`, `transmission_simplified`, `is_accident`, `has_clean_title`.

In [ ]:
CURRENT_YEAR = 2026

df['car_age'] = (CURRENT_YEAR - df['model_year']).apply(lambda x: max(x, 0))

car_age_safe = df['car_age'].replace(0, 1)
df['mileage_per_year'] = df['milage'] / car_age_safe


def extract_horsepower(engine_str):
    """Pulls the HP number out of strings like '300.0HP 3.7L V6...'. Returns NaN if not found."""
    if pd.isna(engine_str):
        return np.nan
    match = re.search(r'(\d+\.?\d*)\s*HP', str(engine_str), re.IGNORECASE)
    return float(match.group(1)) if match else np.nan


def extract_liters(engine_str):
    """Pulls the engine displacement out of strings like '...3.7L V6...'. Returns NaN if not found."""
    if pd.isna(engine_str):
        return np.nan
    match = re.search(r'(\d+\.?\d*)\s*L\b', str(engine_str), re.IGNORECASE)
    return float(match.group(1)) if match else np.nan


df['engine_horsepower'] = df['engine'].apply(extract_horsepower)
df['engine_liters'] = df['engine'].apply(extract_liters)

df['engine_horsepower'] = df['engine_horsepower'].fillna(df['engine_horsepower'].median())
df['engine_liters'] = df['engine_liters'].fillna(df['engine_liters'].median())


def simplify_transmission(t):
    if pd.isna(t):
        return 'Other'
    t = str(t).lower()
    if 'cvt' in t:
        return 'CVT'
    if 'm/t' in t or 'manual' in t:
        return 'Manual'
    if 'a/t' in t or 'automatic' in t:
        return 'Automatic'
    return 'Other'


df['transmission_simplified'] = df['transmission'].apply(simplify_transmission)

df['is_accident'] = df['accident'].apply(
    lambda x: 1 if ('accident' in str(x).lower() and 'none' not in str(x).lower()) else 0
)

df['has_clean_title'] = df['clean_title'].apply(lambda x: 1 if str(x).strip().lower() == 'yes' else 0)

print("New engineered columns:")
display(df[['car_age', 'mileage_per_year', 'engine_horsepower', 'engine_liters',
            'transmission_simplified', 'is_accident', 'has_clean_title']].head())


## Cell 7 — Exploratory Data Analysis *(unchanged)*

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['price'], bins=40, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Distribution of Used Car Prices')
axes[0, 0].set_xlabel('Price ($)')

sns.scatterplot(data=df, x='model_year', y='price', alpha=0.3, ax=axes[0, 1], color='seagreen')
axes[0, 1].set_title('Price vs Model Year')

sns.scatterplot(data=df, x='milage', y='price', alpha=0.3, ax=axes[1, 0], color='indianred')
axes[1, 0].set_title('Price vs Mileage')
axes[1, 0].set_xlabel('Mileage')

top_brands = df['brand'].value_counts().head(10)
sns.barplot(x=top_brands.values, y=top_brands.index, ax=axes[1, 1], color='darkorange')
axes[1, 1].set_title('Top 10 Brands by Number of Listings')
axes[1, 1].set_xlabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
numeric_cols_for_corr = ['price', 'model_year', 'milage', 'car_age', 'mileage_per_year',
                          'engine_horsepower', 'engine_liters', 'is_accident', 'has_clean_title']

plt.figure(figsize=(9, 7))
sns.heatmap(df[numeric_cols_for_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.show()


## Cell 8 — Feature Selection *(MODIFIED — `model` is now included)*

**What changed:** `model` is now added to `categorical_features`. Previously it was excluded
because it has **1,898 unique values** (and 1,082 of those appear only once in the dataset).
It's kept simple by grouping rare models together during encoding in Cell 10, rather than by
excluding the column entirely.

**Target:** `price`

**Categorical features:** `brand`, **`model`** *(new)*, `fuel_type`, `transmission_simplified`,
`ext_col`, `int_col`

**Numerical features:** `model_year`, `milage`, `car_age`, `mileage_per_year`,
`engine_horsepower`, `engine_liters`, `is_accident`, `has_clean_title`


In [ ]:
categorical_features = ['brand', 'model', 'fuel_type', 'transmission_simplified', 'ext_col', 'int_col']
numerical_features = ['model_year', 'milage', 'car_age', 'mileage_per_year',
                       'engine_horsepower', 'engine_liters', 'is_accident', 'has_clean_title']

FEATURE_COLUMNS = categorical_features + numerical_features

X = df[FEATURE_COLUMNS]
y = df['price']

print("Feature columns:", FEATURE_COLUMNS)


## Cell 9 — Train-Test Split *(unchanged)*

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


## Cell 10 — Preprocessing *(MODIFIED — `min_frequency` added to control `model` dimensionality)*

`model` is one-hot encoded along with the other categorical columns, but since 1,082 of the
1,898 distinct models appear only once in the data, encoding every single one would create
hundreds of nearly-useless, sparse columns and risk overfitting. `min_frequency=5` groups any
model (or other rare category) that appears fewer than 5 times into a shared `"infrequent"`
bucket, which keeps the feature space manageable (~225 columns instead of 800+) while still
letting common models like `EcoSport`, `Bronco`, or `Palisade SEL` get their own column and
directly influence the price prediction.

As before, this is wrapped in a `ColumnTransformer` and fit **only on the training data** to
avoid data leakage, and `handle_unknown='ignore'` means an unseen brand/model/color at prediction
time is safely encoded as all-zeros instead of crashing.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=5), categorical_features)
    ],
    remainder='passthrough'
)

print("Preprocessor defined (model column included, min_frequency=5).")


## Cell 11 — Train the XGBoost Model *(unchanged configuration)*

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.08,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb_model)
])

model_pipeline.fit(X_train, y_train)
print("Model training complete (now trained with 'model' as a feature).")


## Cell 12 — Model Evaluation *(unchanged)*

In [ ]:
y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation on Test Set")
print("-" * 35)
print(f"MAE  : ${mae:,.2f}")
print(f"MSE  : {mse:,.2f}")
print(f"RMSE : ${rmse:,.2f}")
print(f"R²   : {r2:.4f}")


## Cell 13 — Actual vs Predicted Price *(unchanged)*

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.4, color='royalblue', edgecolor='k', linewidth=0.2)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs Predicted Used Car Price')
plt.legend()
plt.tight_layout()
plt.show()


## Cell 14 — Feature Importance *(unchanged code — new output includes `model`)*

In [ ]:
feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = model_pipeline.named_steps['regressor'].feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(9, 7))
sns.barplot(data=importance_df, x='importance', y='feature', color='mediumseagreen')
plt.title('Top 15 Most Important Features (XGBoost)')
plt.xlabel('Importance')
plt.ylabel('')
plt.tight_layout()
plt.show()


## Cell 15 — Gradio Application *(MODIFIED — Brand→Model cascading dropdown + INR conversion)*

**What's new in this cell:**

1. **`brand_model_map`** is built directly from the cleaned dataset (`df.groupby('brand')['model'].unique()`)
   — no hard-coded brand/model lists.
2. **Cascading dropdown:** the `Model` dropdown starts populated with the first brand's models.
   A `brand_input.change(...)` event listener updates the `Model` dropdown's choices (and resets
   its selected value) whenever the `Brand` dropdown changes, using `gr.Dropdown(choices=..., value=...)`
   returned from the update function.
3. **`model` is passed into the same fitted `model_pipeline`** used for `brand`, so the exact
   selected model genuinely changes the XGBoost prediction — it isn't a decorative field.
4. **USD → INR conversion:** `predict_price()` runs XGBoost exactly once and returns the numeric
   USD price into a `gr.State()` component. The **₹ Convert to INR** button's callback reads that
   stored number and only does simple arithmetic (`USD * USD_TO_INR`) plus Indian-style comma
   formatting — it never re-runs the model.


In [ ]:
# --- Build the Brand -> Model mapping directly from the cleaned dataset (no hard-coding) ---
brand_model_map = (
    df.groupby('brand')['model']
    .apply(lambda models: sorted(models.unique().tolist()))
    .to_dict()
)

brand_choices = sorted(brand_model_map.keys())
fuel_choices = sorted(df['fuel_type'].dropna().unique().tolist())
transmission_choices = sorted(df['transmission_simplified'].dropna().unique().tolist())
ext_col_choices = df['ext_col'].value_counts().head(30).index.tolist()
int_col_choices = df['int_col'].value_counts().head(30).index.tolist()
accident_choices = ['None reported', 'At least 1 accident or damage reported']
clean_title_choices = ['Yes', 'No']

# Simple, editable, approximate exchange rate (NOT a live rate)
USD_TO_INR = 85.0


def format_inr(amount):
    """Formats a number using the Indian numbering system, e.g. 1572500 -> '15,72,500'."""
    amount = int(round(amount))
    s = str(amount)
    if len(s) <= 3:
        return s
    last3 = s[-3:]
    rest = s[:-3]
    parts = []
    while len(rest) > 2:
        parts.insert(0, rest[-2:])
        rest = rest[:-2]
    if rest:
        parts.insert(0, rest)
    return ','.join(parts) + ',' + last3


def update_model_dropdown(selected_brand):
    """Called when the Brand dropdown changes — refreshes Model choices for that brand only."""
    models_for_brand = brand_model_map.get(selected_brand, [])
    new_value = models_for_brand[0] if models_for_brand else None
    return gr.Dropdown(choices=models_for_brand, value=new_value)


def predict_price(brand, car_model, model_year, milage, fuel_type, engine_text, transmission_text,
                   ext_col, int_col, accident_text, clean_title_text):
    """Runs the SAME cleaning + feature engineering used in training, then predicts ONCE.
    Returns (display_text, numeric_usd_price_for_state)."""
    try:
        model_year = float(model_year)
        milage = float(milage)

        car_age = max(CURRENT_YEAR - model_year, 0)
        car_age_safe = car_age if car_age != 0 else 1
        mileage_per_year = milage / car_age_safe

        engine_hp = extract_horsepower(engine_text)
        engine_l = extract_liters(engine_text)
        if pd.isna(engine_hp):
            engine_hp = df['engine_horsepower'].median()
        if pd.isna(engine_l):
            engine_l = df['engine_liters'].median()

        transmission_simplified = simplify_transmission(transmission_text)
        is_accident = 1 if 'accident' in accident_text.lower() and 'none' not in accident_text.lower() else 0
        has_clean_title = 1 if clean_title_text.strip().lower() == 'yes' else 0

        input_row = pd.DataFrame([{
            'brand': brand,
            'model': car_model,                       # <-- model now included in the pipeline input
            'fuel_type': fuel_type,
            'transmission_simplified': transmission_simplified,
            'ext_col': ext_col,
            'int_col': int_col,
            'model_year': model_year,
            'milage': milage,
            'car_age': car_age,
            'mileage_per_year': mileage_per_year,
            'engine_horsepower': engine_hp,
            'engine_liters': engine_l,
            'is_accident': is_accident,
            'has_clean_title': has_clean_title
        }])

        predicted_price_usd = float(model_pipeline.predict(input_row)[0])
        display_text = f"Estimated Used Car Price:\n${predicted_price_usd:,.0f}"

        # Return the display text, the raw numeric USD value (stored in gr.State),
        # and reset/hide the INR output until the user clicks the convert button.
        return display_text, predicted_price_usd, "", gr.Button(visible=True)

    except Exception as e:
        return f"Could not generate a prediction. Please check your inputs. ({e})", None, "", gr.Button(visible=False)


def convert_to_inr(usd_price):
    """Converts the ALREADY-PREDICTED USD price to INR. Does NOT call the ML model again."""
    if usd_price is None:
        return "Please predict a price first."
    inr_price = usd_price * USD_TO_INR
    return (
        f"Estimated Used Car Price in INR:\n₹{format_inr(inr_price)}\n"
        f"(Approximate, based on a fixed rate of 1 USD = ₹{USD_TO_INR:.1f}. Not a live exchange rate.)"
    )


with gr.Blocks(title="Used Car Price Predictor") as demo:
    gr.Markdown("# 🚗 Used Car Price Prediction\nSelect a Brand, then a Model, fill in the rest, and click **Predict Price**.")

    with gr.Row():
        with gr.Column():
            brand_in = gr.Dropdown(choices=brand_choices, label="Brand", value=brand_choices[0])
            model_in = gr.Dropdown(choices=brand_model_map[brand_choices[0]], label="Car Model",
                                    value=brand_model_map[brand_choices[0]][0])
            model_year_in = gr.Number(label="Model Year", value=2020)
            milage_in = gr.Number(label="Mileage (miles)", value=30000)
            fuel_in = gr.Dropdown(choices=fuel_choices, label="Fuel Type", value=fuel_choices[0])
            engine_in = gr.Textbox(label="Engine (e.g. '300.0HP 3.7L V6 Cylinder Engine')",
                                    value="200.0HP 2.5L 4 Cylinder Engine Gasoline Fuel")
        with gr.Column():
            transmission_in = gr.Dropdown(choices=transmission_choices, label="Transmission",
                                           value=transmission_choices[0])
            ext_col_in = gr.Dropdown(choices=ext_col_choices, label="Exterior Color", value=ext_col_choices[0])
            int_col_in = gr.Dropdown(choices=int_col_choices, label="Interior Color", value=int_col_choices[0])
            accident_in = gr.Dropdown(choices=accident_choices, label="Accident History",
                                       value=accident_choices[0])
            clean_title_in = gr.Dropdown(choices=clean_title_choices, label="Clean Title",
                                          value=clean_title_choices[0])

    # Cascading dropdown: changing Brand refreshes the Model choices
    brand_in.change(fn=update_model_dropdown, inputs=brand_in, outputs=model_in)

    predict_btn = gr.Button("Predict Price", variant="primary")
    usd_output_box = gr.Textbox(label="Prediction Result (USD)")
    usd_price_state = gr.State(value=None)   # stores the numeric USD prediction, nothing else

    convert_btn = gr.Button("₹ Convert to INR", visible=False)
    inr_output_box = gr.Textbox(label="Prediction Result (INR)")

    predict_btn.click(
        fn=predict_price,
        inputs=[brand_in, model_in, model_year_in, milage_in, fuel_in, engine_in, transmission_in,
                ext_col_in, int_col_in, accident_in, clean_title_in],
        outputs=[usd_output_box, usd_price_state, inr_output_box, convert_btn]
    )

    convert_btn.click(
        fn=convert_to_inr,
        inputs=usd_price_state,
        outputs=inr_output_box
    )

demo.launch(share=True)
